In [2]:
import tensorflow as tf
from tensorflow.keras.models import load_model
import pickle
import pandas as pd
import numpy as np

In [4]:
# Load model
model = load_model('model.h5')

# Load encoders
with open('onehot_encoder_geo.pkl', 'rb') as file:
    onehot_encoder_geo = pickle.load(file)

with open('label_encoder_gender.pkl', 'rb') as file:
    label_encoder_gender = pickle.load(file)

# Load scaler
with open('scaler.pkl', 'rb') as file:
    scaler = pickle.load(file)

In [5]:
input_data = {
    'CreditScore': 600,
    'Geography': 'France',
    'Gender': 'Male',
    'Age': 40,
    'Tenure': 3,
    'Balance': 60000,
    'NumOfProducts': 2,
    'HasCrCard': 1,
    'IsActiveMember': 1,
    'EstimatedSalary': 50000
}

In [6]:
input_df = pd.DataFrame([input_data])

In [7]:
input_df['Gender'] = label_encoder_gender.transform(input_df['Gender'])

In [8]:
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,1,40,3,60000,2,1,1,50000


In [9]:
geo_encoded = onehot_encoder_geo.transform(input_df[['Geography']]).toarray()

geo_df = pd.DataFrame(
    geo_encoded,
    columns=onehot_encoder_geo.get_feature_names_out(['Geography'])
)

In [10]:
geo_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [11]:
input_df = input_df.drop('Geography', axis=1)

In [12]:
input_df = pd.concat([input_df, geo_df], axis=1)

In [13]:
input_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,1,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [14]:
## scaliing the input data
input_scaled = scaler.transform(input_df)

In [15]:
input_scaled

array([[-0.53598516,  0.91324755,  0.10479359, -0.69539349, -0.25781119,
         0.80843615,  0.64920267,  0.97481699, -0.87683221,  1.00150113,
        -0.57946723, -0.57638802]])

In [16]:
# Predict Churn
prediction = model.predict(input_scaled)

1/1 [==============================] - 1s 662ms/step


In [17]:
prediction

array([[0.07860111]], dtype=float32)

In [18]:
prediction = (prediction > 0.5)

In [19]:
prediction

array([[False]])

In [20]:
if prediction[0][0] == 1:
    print("Customer will churn ❌")
else:
    print("Customer will stay ✔")

Customer will stay ✔
